# Demo - The Function-Calling Loop
**Day 2 - Session 4**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/demos/notebooks/demo-function-calling.ipynb)

**Goal:** Show the mechanics of function calling: the model requests a tool by
name with arguments, your code runs it, and the result goes back to the model.

We simulate the model with a small rule-based function so the loop runs
offline with no API key. Real providers return the same shape (a tool name
plus JSON arguments); only the "decide" step is done by the LLM instead of our
stub.

No API key needed. No model download. Pure Python.

In [ ]:
import json

# 1. Tools your code exposes. The model only sees names, docs, and schemas.
TOOL_SCHEMA = [
    {"name": "get_weather", "description": "Current weather for a city.",
     "parameters": {"city": "string"}},
    {"name": "calculator", "description": "Evaluate a simple math expression.",
     "parameters": {"expression": "string"}},
]

## 1. The tools
Two toy tools. `get_weather` looks up a fixed table, `calculator` evaluates a
trusted math expression (demo only).

In [ ]:
def get_weather(city):
    fake = {"Paris": "18C and cloudy", "Austin": "33C and sunny"}
    return fake.get(city, "unknown")


def calculator(expression):
    return eval(expression, {"__builtins__": {}}, {})  # demo only: trusted input


TOOLS = {"get_weather": get_weather, "calculator": calculator}

## 2. A stand-in for the model
In production, the LLM decides whether to call a tool and with what
arguments. Here a rule-based stub plays that role so the demo is deterministic
and offline.

In [ ]:
def fake_model(user_message):
    """Stand-in for an LLM. Returns either a tool call or a final answer."""
    text = user_message.lower()
    if "weather" in text and "paris" in text:
        return {"type": "tool_call", "name": "get_weather",
                "arguments": {"city": "Paris"}}
    if "15" in text and "*" not in text:  # "15 percent tip on 40"
        return {"type": "tool_call", "name": "calculator",
                "arguments": {"expression": "40 * 0.15"}}
    return {"type": "final", "content": "I can help with weather or math."}

## 3. Run the loop
Ask the model, run the requested tool if any, and print the final reply.

In [ ]:
def run(user_message):
    print(f"\nUser: {user_message}")
    step = fake_model(user_message)

    if step["type"] == "tool_call":
        name, args = step["name"], step["arguments"]
        print(f"  Model requests tool: {name}({json.dumps(args)})")
        result = TOOLS[name](**args)
        print(f"  Your code runs it   -> {result}")
        print(f"  Result sent back to the model, which writes the final reply.")
        print(f"  Model: The answer is {result}.")
    else:
        print(f"  Model: {step['content']}")


print(f"Tools exposed to the model:\n{json.dumps(TOOL_SCHEMA, indent=2)}")

run("What is the weather in Paris?")
run("What is a 15 percent tip on a 40 dollar bill?")

## Expected output
- The Paris weather question triggers a `get_weather` tool call and returns
  `18C and cloudy`.
- The tip question triggers a `calculator` tool call evaluating `40 * 0.15`
  and returns `6.0`.

**Takeaway:** the model never runs code. It only requests a call in a
structured format; your code decides what actually executes.